In [ ]:
# Script plots the change in the interannual standard deviation of extreme heat season length between periods in the ERA5 data. 
# Uses centered means (subtract mean from the yearly length values before calculating standard deviation...this is key given the permutation test).
# It can be used to re-create the change in standard deviation figures in the manuscript.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
import re
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

In [ ]:
# Load in the ERA5 Heat Season Characteristics Files for the two time periods of interest.
# Script is designed to work with one temperature variable (TMAX or TMIN) at a time.

ds1 = xr.load_dataset("/...baseline period TMAX file...")
ds2 = xr.load_dataset("/...comparison period TMAX file...")


In [ ]:
# Grab the array of analytical year start months (see Methods in the paper).
trop_map_p1 = ds1['ref_trop_start_month']

In [ ]:
# I CALCULATE SIGNIFICANCE OF CHANGE AND CALL THE FDR PROCEDURE FROM WITHIN THE VARIABILITY CALCULATION FUNCTION. SO NEED TO DEFINE
# THIS FDR FUNCTION HERE.

In [ ]:
def apply_fdr_control(p_map, alpha=0.05, method='indep'):
    """
    Applies the False Discovery Rate (FDR) control to a map of p-values
    following the Benjamini-Hochberg procedure (Wilks, 2016).
    """
    print(f"Applying FDR Control (alpha={alpha})...")
    
    # 1. Flatten the map to 1D array
    p_values = p_map.values.flatten()
    valid_mask = ~np.isnan(p_values)
    p_valid = p_values[valid_mask]
    
    N = len(p_valid) # Total number of valid tests
    
    # 2. Sort P-values (smallest to largest)
    sorted_p = np.sort(p_valid)
    
    # 3. Calculate Critical Values
    k = np.arange(1, N + 1)
    
    if method == 'dep':
        c_N = np.sum(1 / k)
        p_crit = (k / (N * c_N)) * alpha
    else:
        p_crit = (k / N) * alpha
        
    # 4. Find Cutoff
    is_below = sorted_p <= p_crit
    
    if np.any(is_below):
        max_k_idx = np.where(is_below)[0].max()
        p_threshold = sorted_p[max_k_idx]
        
        print(f"  > FDR Threshold found: p <= {p_threshold:.5f}")
        print(f"  > (Standard p=0.05 threshold would be less strict)")
    else:
        print("  > No p-values passed FDR control.")
        p_threshold = 0.0
        
    # 5. Create Significance Mask
    sig_mask = p_map <= p_threshold
    
    return sig_mask

In [ ]:
# THIS FUNCTION WILL CALCULATE THE REFERENCE PERIOD STANDARD DEVIATION AS WELL AS THE CHANGE IN STANDARD DEVIATION
# FOR THE CHANGE, I THINK IT IS IMPORTANT TO FIRST CENTER THE VALUES (SUBTRACT MEAN). NOTE, THIS DOESN'T IMPACT THE REF PERIOD CALCULATION

In [ ]:
def calculate_variability_change_centered(length_ref, length_comp, n_permutations=1000, alpha=0.05):
    """
    Calculates the standard deviation of season length for both periods, 
    the observed difference, and tests for significance using permutations
    and FDR control.
    """
    print("1. Calculating Standard Deviations & Centering Data...")
    
    # Calculate interannual variability (Standard Deviation)
    std_ref = length_ref.std(dim='year', skipna=True)
    std_comp = length_comp.std(dim='year', skipna=True)
    
    # Calculate the observed change in variability
    obs_diff_std = std_comp - std_ref
    
    # --- CENTER THE DATA ---
    # Subtract the mean from each period to isolate the variance 
    # before we shuffle them together.
    mean_ref = length_ref.mean(dim='year', skipna=True)
    mean_comp = length_comp.mean(dim='year', skipna=True)
    
    centered_ref = length_ref - mean_ref
    centered_comp = length_comp - mean_comp
    
    print(f"2. Running Vectorized Permutation Test ({n_permutations} shuffles)...")
    
    # --- Set a random seed for complete reproducibility ---
    np.random.seed(42)
    
    # Concatenate the CENTERED data, not the raw data!
    combined = xr.concat([centered_ref, centered_comp], dim='year')
    n_ref = length_ref.sizes['year']
    n_total = combined.sizes['year']
    
    vals = combined.values
    obs_diff_vals = obs_diff_std.values
    exceed_count = np.zeros_like(obs_diff_vals)
    
    for _ in range(n_permutations):
        # Shuffle the 60 years of centered data
        idx = np.random.permutation(n_total)
        shuffled = vals[idx, ...]
        
        # Split into pseudo-reference and pseudo-comparison periods
        pseudo_ref = shuffled[:n_ref, ...]
        pseudo_comp = shuffled[n_ref:, ...]
        
        # Calculate the pseudo-difference in STANDARD DEVIATION
        with np.errstate(invalid='ignore'): 
            pseudo_diff = np.nanstd(pseudo_comp, axis=0) - np.nanstd(pseudo_ref, axis=0)
        
        # Two-Tailed Test: Does random shuffling create a larger shift in variance?
        exceed_count += (np.abs(pseudo_diff) >= np.abs(obs_diff_vals))
        
    # 3. Calculate raw p-values and format as an xarray DataArray
    p_values_vals = exceed_count / n_permutations
    p_val_map = xr.DataArray(p_values_vals, coords=obs_diff_std.coords, dims=obs_diff_std.dims)
    
    # 4. Apply False Discovery Rate Control
    is_significant = apply_fdr_control(p_val_map, alpha=alpha)
    
    print("Test Complete.")
    return std_ref, obs_diff_std, is_significant

In [ ]:
# 1. Run the test
std_ref, diff_std, sig_mask_std = calculate_variability_change_centered(
    ds1['season_length'], 
    ds2['season_length'], 
    n_permutations=1000,
    alpha=0.05
)

In [ ]:
# PLOT THE REFERENCE PERIOD VARIABILITY AND THE CHANGE IN VARIABILITY

In [ ]:
def map_reference_variability(std_ref):
    """
    Plots the baseline standard deviation of extreme heat season length.
    """
    print("Generating Reference Map...")
    
    # Initialize the figure with standard screen sizing
    fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree()})

    
    ocean_mask = cfeature.OCEAN.with_scale('110m') 
    coastlines = cfeature.COASTLINE.with_scale('110m')
    #borders = cfeature.BORDERS.with_scale('110m')
    
    # --- Plot Data ---
    plot1 = std_ref.plot(
        ax=ax, transform=ccrs.PlateCarree(), 
        cmap='YlOrRd', 
        levels=np.arange(0, 100, 10),
        add_colorbar=False
    )
    
    # --- Apply Visual Masks ---
    ax.add_feature(ocean_mask, facecolor='white', zorder=2)
    ax.add_feature(coastlines, linewidth=0.8, edgecolor='black', zorder=3)
    #ax.add_feature(borders, linewidth=0.4, linestyle=':', edgecolor='gray', zorder=3)
    
    #ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--', zorder=0)

    # --- NEW: Gridlines ---
    # Set zorder to 4 so they appear over the white ocean mask
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    
    # This forces the labels to only appear on the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    
    # --- Formatting ---
    ax.set_title("1966-1995 TMAX Standard Deviation of Season Length")
    #ax.set_title("1966-1995 TMIN Standard Deviation of Season Length")
    
    cbar1 = plt.colorbar(plot1, ax=ax, orientation='horizontal', pad=0.08, aspect=30, extend='max')
    cbar1.set_label('Standard Deviation (Days)')

    #plt.savefig("plots/ERA5_TMAX_SeasonLength_Variability_1966-1995.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonLength_Variability_1966-1995.pdf", format="pdf", bbox_inches="tight")
    plt.tight_layout()
    plt.show()
    
    return std_ref

In [ ]:
std_ref = map_reference_variability(std_ref)

In [ ]:
def map_variability_change(obs_diff_std, is_significant):
    """
    Plots the significant change in standard deviation between periods, 
    masking non-significant land and oceans white.
    """
    print("Applying significance mask...")
    masked_diff = obs_diff_std.where(is_significant)
    
    print("Generating Change Map...")
    
    # Initialize the figure with standard screen sizing
    #fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.Robinson()})
    fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    
    ocean_mask = cfeature.OCEAN.with_scale('110m') 
    coastlines = cfeature.COASTLINE.with_scale('110m')
    #borders = cfeature.BORDERS.with_scale('110m')
    
    # --- Plot Data ---
    plot2 = masked_diff.plot(
        ax=ax, transform=ccrs.PlateCarree(), 
        cmap='PuOr_r', 
        levels=np.arange(-15, 18, 3),
        extend='both',
        add_colorbar=False 
    )
    
    # --- Apply Visual Masks ---
    ax.add_feature(ocean_mask, facecolor='white', zorder=2)
    ax.add_feature(coastlines, linewidth=0.8, edgecolor='black', zorder=3)
    #ax.add_feature(borders, linewidth=0.4, linestyle=':', edgecolor='gray', zorder=3)
    
    #ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--', zorder=0)

    # Gridlines 
    # Set zorder to 4 so they appear over the white ocean mask
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    
    # This forces the labels to only appear on the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    
    # --- Formatting ---
    ax.set_title("Change in TMAX Standard Deviation (1996-2025 - 1966-1995)\n(FDR Significant)")
    #ax.set_title("Change in TMIN Standard Deviation (1996-2025 - 1966-1995)\n(FDR Significant)")

    
    cbar2 = plt.colorbar(plot2, ax=ax, orientation='horizontal', pad=0.08, aspect=30)
    cbar2.set_label('Change in Standard Deviation')


    plt.savefig("plots/ERA5_TMAX_SeasonLength_ChangeinVariability_1996-2025_1966-1995.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonLength_ChangeinVariability_1996-2025_1966-1995.pdf", format="pdf", bbox_inches="tight")
    plt.tight_layout()
    plt.show()
    
    return masked_diff

In [ ]:
masked_diff_std = map_variability_change(diff_std, sig_mask_std)